# Airlines Use-Case — End-to-End Data Pipeline

  
**Version:** 1.0.0  
**Purpose:** Ingest → Clean → Transform → Validate → Aggregate raw flight / booking data  
and produce a cleaned dataset ready for Power BI visualisation.

---

## Table of Contents
1. [Setup & Configuration](#1-Setup-&-Configuration)
2. [Ingestion Layer](#2-Ingestion-Layer)
3. [Flight Data Cleaning](#3-Flight-Data-Cleaning)
4. [Booking Data Cleaning](#4-Booking-Data-Cleaning)
5. [Passenger Data (PII Protection)](#5-Passenger-Data-(PII-Protection))
6. [Transformation & Fact Table Join](#6-Transformation-&-Fact-Table-Join)
7. [KPI Aggregations](#7-KPI-Aggregations)
8. [Export Cleaned Data](#8-Export-Cleaned-Data)
9. [Pipeline Summary](#9-Pipeline-Summary)


---
## 1. Setup & Configuration

Import all required libraries, configure logging, and define file paths.  
The pipeline uses only standard Python libraries and `pandas`/`numpy` — no external  
orchestration framework is required, making it portable across any Python environment.


In [ ]:
import os, sys, warnings, logging, datetime
from pathlib import Path

import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

# ── Logging ──────────────────────────────────────────────────────────────
LOG_DIR = Path("../logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)
log_file = LOG_DIR / f"pipeline_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    handlers=[
        logging.FileHandler(log_file, encoding="utf-8"),
        logging.StreamHandler(sys.stdout),
    ],
)
logger = logging.getLogger("AirlinesPipeline")

# ── Paths ─────────────────────────────────────────────────────────────────
INPUT_DIR  = Path("/mnt/user-data/uploads")
OUTPUT_DIR = Path("../cleaned_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILES = {
    "flights"    : INPUT_DIR / "UseCase_-_Airlines.xlsx",
    "bookings"   : INPUT_DIR / "dim_bookings.csv",
    "passengers" : INPUT_DIR / "dim_passengers_masked.csv",
}

quality_log = []   # accumulates data-quality rule results

def record_dq(rule, n):
    quality_log.append({"rule": rule, "affected_rows": n})
    logger.info(f"  DQ | {rule}: {n} rows")

logger.info("Setup complete. Paths configured.")


---
## 2. Ingestion Layer

### 2a. Flights — Excel Workbook
The raw flight data lives in the `flights` sheet of the `.xlsx` workbook.  
`openpyxl` parses `departure_time` and `arrival_time` directly as datetime objects.  
The `duration` column is stored as a `datetime.time` object (HH:MM:SS) and is  
converted to fractional minutes for computation.

### 2b. Bookings — CSV
Flat CSV with one booking per row. Fields: `booking_id`, `passenger_id`,  
`flight_id`, `booking_date`, `status`.

### 2c. Passengers — CSV (Masked)
Passenger PII is already hashed at source. Only non-identifying fields are  
loaded: `passenger_id` (internal join key), `passenger_key` (hash), `age_band`,  
`gender`.


In [ ]:
import datetime as dt

def ingest_flights(path):
    logger.info(f"Reading flights from: {path}")
    df = pd.read_excel(path, sheet_name="flights", engine="openpyxl")
    logger.info(f"  → {len(df):,} raw flight rows ingested")

    # Convert duration (datetime.time) → minutes
    def time_to_minutes(t):
        if pd.isna(t) or t is None: return np.nan
        if isinstance(t, str):
            parts = t.split(":")
            return int(parts[0])*60 + int(parts[1]) + int(parts[2])/60
        if isinstance(t, dt.time):
            return t.hour*60 + t.minute + t.second/60
        return np.nan

    df["duration_minutes_raw"] = df["duration"].apply(time_to_minutes).round(2)
    logger.info("  Converted duration → minutes")
    return df

def ingest_csv(path, label):
    logger.info(f"Reading {label} from: {path}")
    df = pd.read_csv(path)
    logger.info(f"  → {len(df):,} rows ingested")
    return df

raw_flights    = ingest_flights(RAW_FILES["flights"])
raw_bookings   = ingest_csv(RAW_FILES["bookings"],   "bookings")
raw_passengers = ingest_csv(RAW_FILES["passengers"], "passengers")

print(f"\nRaw flights shape  : {raw_flights.shape}")
print(f"Raw bookings shape : {raw_bookings.shape}")
print(f"Raw passengers     : {raw_passengers.shape}")


In [ ]:
# Preview raw flights
raw_flights.head(3)


---
## 3. Flight Data Cleaning

The following cleaning rules are applied in order.  
Each rule is recorded in the **Data Quality Log** for audit traceability.

| Step | Rule | Rationale |
|------|------|-----------|
| 3a | Normalise column names | Consistent snake_case |
| 3b | Fill blank airline as 'Unknown' | Retain rows; flag for investigation |
| 3c | Remove exact duplicate rows | Deduplicate data |
| 3d | Reject malformed flight IDs | Must match pattern `[A-Z0-9]{3-7}` |
| 3e | Reject same-origin/destination | Logically invalid flights |
| 3f | Reject missing timestamps | Cannot compute duration |
| 3g | Detect & correct overnight flights | arrival < departure → add +1 day |
| 3h | Recalculate duration from timestamps | Authoritative computed field |
| 3i | Reject duration ≤ 0 or > 720 min | Business rule: max 12-hour domestic |
| 3j | Remove conflicting schedules | Same flight_id + time → keep first |
| 3k | Derive route | `source-destination` string |


In [ ]:
flights = raw_flights.copy()
flights.columns = [c.strip().lower().replace(" ", "_") for c in flights.columns]

# 3b — Missing airline
missing_airline = flights["airline"].isna() | (flights["airline"].astype(str).str.strip() == "")
flights.loc[missing_airline, "airline"] = "Unknown"
record_dq("Missing or unknown airline labelled Unknown", int(missing_airline.sum()))

# 3c — Duplicates
n_before = len(flights)
flights = flights.drop_duplicates()
record_dq("Exact duplicate flight rows removed", n_before - len(flights))

# 3d — Malformed IDs
malformed = ~flights["flight_id"].astype(str).str.match(r"^[A-Z0-9]{3,7}$", na=False)
record_dq("Malformed flight identifiers rejected", int(malformed.sum()))
flights = flights[~malformed].copy()

# 3e — Same origin/destination
same_od = flights["source"] == flights["destination"]
record_dq("Same-origin and destination flights rejected", int(same_od.sum()))
flights = flights[~same_od].copy()

# 3f — Missing timestamps
ts_invalid = flights["departure_time"].isna() | flights["arrival_time"].isna()
record_dq("Missing or invalid timestamps rejected", int(ts_invalid.sum()))
flights = flights[~ts_invalid].copy()

# 3g — Overnight correction
overnight_mask = flights["arrival_time"] < flights["departure_time"]
flights.loc[overnight_mask, "arrival_time"] += pd.Timedelta(days=1)
record_dq("Overnight arrival dates corrected", int(overnight_mask.sum()))
flights["overnight_flight"] = overnight_mask

# 3h — Recalculate duration
flights["duration_minutes"] = (
    (flights["arrival_time"] - flights["departure_time"])
    .dt.total_seconds() / 60
).round(2)

# 3i — Invalid duration
invalid_dur = (flights["duration_minutes"] <= 0) | (flights["duration_minutes"] > 720)
record_dq("Nonpositive or over 12-hour durations rejected", int(invalid_dur.sum()))
flights = flights[~invalid_dur].copy()

# 3j — Conflicting schedules
dup_sched = flights.duplicated(subset=["flight_id","departure_time","source","destination"], keep="first")
record_dq("Conflicting repeated flight schedules rejected", int(dup_sched.sum()))
flights = flights[~dup_sched].copy()

# 3k — Route
flights["route"] = flights["source"] + "-" + flights["destination"]
flights["flight_record_status"] = "VALID"

print(f"Flights after cleaning: {len(flights):,}")
flights[["flight_id","airline","route","departure_time","arrival_time","duration_minutes","overnight_flight"]].head(5)


---
## 4. Booking Data Cleaning

**Booking status standardisation:**  
Valid statuses are `CONFIRMED`, `CANCELLED`, `PENDING`.  
Any other value (including nulls) is relabelled `UNKNOWN` and flagged.

**Booking date:**  
Parsed to `datetime64` for temporal joins and trend analysis.


In [ ]:
bookings = raw_bookings.copy()
bookings.columns = [c.strip().lower().replace(" ", "_") for c in bookings.columns]

bookings["booking_date"] = pd.to_datetime(bookings["booking_date"], errors="coerce")

VALID_STATUSES = {"CONFIRMED", "CANCELLED", "PENDING"}
bookings["status"] = bookings["status"].astype(str).str.upper().str.strip()
invalid_status = ~bookings["status"].isin(VALID_STATUSES)
bookings.loc[invalid_status, "status"] = "UNKNOWN"
record_dq("Missing or invalid booking status labelled UNKNOWN", int(invalid_status.sum()))

bookings["is_confirmed"] = bookings["status"] == "CONFIRMED"

print(f"Bookings after cleaning: {len(bookings):,}")
print("\nStatus distribution:")
print(bookings["status"].value_counts())


---
## 5. Passenger Data (PII Protection)

Per data governance policy, **no personally identifiable information is loaded**.  
The source already hashes passenger names and contact details into `passenger_key`.  
We retain only the four safe columns needed for demographic segmentation.


In [ ]:
passengers = raw_passengers.copy()
passengers.columns = [c.strip().lower().replace(" ", "_") for c in passengers.columns]

PII_SAFE = ["passenger_id", "passenger_key", "age_band", "gender"]
passengers = passengers[[c for c in PII_SAFE if c in passengers.columns]]

print(f"Passengers (PII-safe): {len(passengers):,}")
print("Age band distribution:")
print(passengers["age_band"].value_counts())


---
## 6. Transformation & Fact Table Join

### Data Model
```
dim_bookings  ──┐
                ├──> fact_flight_operations <── dim_passengers_masked
dim_flights   ──┘
```

**Join strategy:** `bookings LEFT JOIN flights ON flight_id`  
- Unmatched bookings (no corresponding flight record) are tagged `REJECTED`  
  in `flight_match_status` and retained for audit — they are **not dropped**.

**Payment amounts:**  
- `CONFIRMED` and `PENDING` bookings carry a payment amount drawn from  
  a reproducible random seed (₹3,000 – ₹25,000 per booking).  
- `CANCELLED` and `UNKNOWN` bookings have `payment_amount = 0`.


In [ ]:
rng = np.random.default_rng(seed=42)

fact = bookings.merge(
    flights[["flight_id","airline","source","destination","route",
             "departure_time","arrival_time","duration_minutes",
             "overnight_flight","flight_record_status"]],
    on="flight_id", how="left", indicator=True,
)

fact["flight_match_status"] = np.where(fact["_merge"] == "both", "VALID", "REJECTED")
unmatched = int((fact["_merge"] != "both").sum())
record_dq("Unmatched or rejected bookings", unmatched)
fact.drop(columns=["_merge"], inplace=True)

for col in ["airline","source","destination","route","flight_record_status"]:
    fact[col] = fact[col].fillna("UNKNOWN")
fact["duration_minutes"] = fact["duration_minutes"].fillna(0.0)
fact["overnight_flight"]  = fact["overnight_flight"].fillna(False)

n = len(fact)
fact["payment_amount"] = np.where(
    fact["status"].isin(["CONFIRMED","PENDING"]),
    rng.uniform(3_000, 25_000, size=n).round(2), 0.0)
fact["payment_rows"] = np.where(
    fact["status"].isin(["CONFIRMED","PENDING"]),
    rng.integers(1, 4, size=n), 0)

fact = fact.merge(passengers, on="passenger_id", how="left")

print(f"Fact table: {fact.shape[0]:,} rows × {fact.shape[1]} columns")
fact.head(3)


---
## 7. KPI Aggregations

### Overview KPIs
Single-row summary of the entire dataset — used for the dashboard header tiles.

### KPI by Airline
Aggregated booking count, average duration, overnight count, and revenue per carrier.

### KPI by Route
Aggregated metrics per route, including confirmation rate and revenue.


In [ ]:
# Overview
kpi_overview = pd.DataFrame([
    {"metric": "Total bookings",                "value": len(fact)},
    {"metric": "Confirmed bookings",            "value": int(fact["is_confirmed"].sum())},
    {"metric": "Valid flight records linked",   "value": int((fact["flight_match_status"]=="VALID").sum())},
    {"metric": "Average flight duration minutes","value": round(float(fact.loc[fact["duration_minutes"]>0,"duration_minutes"].mean()),2)},
    {"metric": "Overnight flights",             "value": int(fact["overnight_flight"].sum())},
    {"metric": "Unmatched or rejected bookings","value": int((fact["flight_match_status"]=="REJECTED").sum())},
    {"metric": "Payment revenue",               "value": round(float(fact["payment_amount"].sum()),2)},
])
kpi_overview


In [ ]:
# By Airline
kpi_airline = (
    fact[fact["flight_match_status"]=="VALID"]
    .groupby("airline", as_index=False)
    .agg(booking_count=("booking_id","count"),
         average_duration_minutes=("duration_minutes","mean"),
         overnight_flights=("overnight_flight","sum"),
         payment_revenue=("payment_amount","sum"))
)
kpi_airline["average_duration_minutes"] = kpi_airline["average_duration_minutes"].round(1)
kpi_airline["payment_revenue"] = kpi_airline["payment_revenue"].round(2)
kpi_airline.sort_values("booking_count", ascending=False)


In [ ]:
# By Route
kpi_route = (
    fact[fact["flight_match_status"]=="VALID"]
    .groupby(["route","source","destination"], as_index=False)
    .agg(booking_count=("booking_id","count"),
         average_duration_minutes=("duration_minutes","mean"),
         confirmed_bookings=("is_confirmed","sum"),
         payment_revenue=("payment_amount","sum"))
)
kpi_route["average_duration_minutes"] = kpi_route["average_duration_minutes"].round(1)
kpi_route["payment_revenue"] = kpi_route["payment_revenue"].round(2)
kpi_route.sort_values("booking_count", ascending=False).head(10)


---
## 8. Export Cleaned Data

All output files are saved to `../cleaned_data/` as UTF-8 CSV.  
These files are ready for direct import into Power BI.


In [ ]:
exports = {
    "fact_flight_operations.csv"  : fact,
    "dim_flights.csv"             : flights,
    "dim_bookings.csv"            : bookings,
    "dim_passengers_masked.csv"   : passengers,
    "kpi_overview.csv"            : kpi_overview,
    "kpi_by_airline.csv"          : kpi_airline,
    "kpi_by_route.csv"            : kpi_route,
    "data_quality_log.csv"        : pd.DataFrame(quality_log),
}

for fname, df in exports.items():
    path = OUTPUT_DIR / fname
    df.to_csv(path, index=False)
    print(f"  {fname}  ({len(df):,} rows × {df.shape[1]} cols)")


---
## 9. Pipeline Summary

The table below consolidates all data quality decisions made during this run.


In [ ]:
dq_df = pd.DataFrame(quality_log)
print("=" * 60)
print("  DATA QUALITY LOG")
print("=" * 60)
print(dq_df.to_string(index=False))
print()
print(f"Raw flights ingested     : {len(raw_flights):,}")
print(f"Cleaned flights          : {len(flights):,}")
print(f"Bookings                 : {len(bookings):,}")
print(f"Passengers (masked)      : {len(passengers):,}")
print(f"Fact table rows          : {len(fact):,}")
print(f"Confirmed bookings       : {int(fact['is_confirmed'].sum()):,}")
print(f"Overnight flights        : {int(fact['overnight_flight'].sum())}")
print(f"Total payment revenue    : ₹{fact['payment_amount'].sum():,.2f}")
print()
print("Pipeline completed.")
